<Strong><span style = "font-size:50px;color:orange;underline:true">Finding Primes</strong><br>
- prime numbers dont currently have a prediction equation<br>
- the closest know is "Eulers rule" expressed as n^2 - n + 41 whick works where n<40 <br>
<br>This project aims to use eulers rule as a starting point to explore other avenues to fine a more accurate mothod to predict prime numbers through the implimentation of genetic programming.

In [14]:
print("Starting Imports...")
import math 
import numpy as np
import operator
from deap import base, gp, creator, tools, algorithms
import itertools
import random
print ("Imports Completed.")

Starting Imports...
Imports Completed.


<span style = "font-size:30px;color:Yellow">Defining the Primitives - Operations

In [15]:
print("Starting DEAP Setup with base...")

def protectedDiv(left, right):
    try:
        return left / right
    except ZeroDivisionError:
        return 1.0

# You can add a power function safely
def safe_power(a, b):
    try:
        # Clamp both values to reasonable ranges
        a = max(min(a, 1e6), -1e6)
        b = int(round(max(min(b, 10), -10)))  # limit exponent to ±10
        return a ** b
    except (OverflowError, ValueError, ZeroDivisionError):
        return 0

# def rand_const(random_obj):
#     return random.uniform(1, 1000)

pset = gp.PrimitiveSet("MAIN", 1)
pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(protectedDiv, 2, name="div")

# Use zero-arg lambda for ephemeral constant:
pset.addEphemeralConstant("randConst", lambda: random.randint(1, 1000))

pset.addTerminal(1)
pset.addTerminal(2)
pset.addTerminal(41)
pset.addPrimitive(safe_power, 2)
pset.renameArguments(ARG0="n")

# Add a power primitive for exponents
pset.addPrimitive(safe_power, 2)

pset.renameArguments(ARG0="n")

# Recreate Euler's formula using the new primitives
euler_formula_tree = gp.PrimitiveTree.from_string("add(add(safe_power(n, 2), n), 41)", pset)

print("Base Set-up Completed.")

Starting DEAP Setup with base...
Base Set-up Completed.


<span style = "font-size:30px;color:Yellow">Fitness Function

In [ ]:
def is_prime(num):
    """Helper function to check for primality."""
    num = int(abs(num))
    if num <= 1:
        return False
    if num <= 3:
        return True
    if num % 2 == 0 or num % 3 == 0:
        return False
    i = 5
    while i * i <= num:
        if num % i == 0 or num % (i + 2) == 0:
            return False
        i += 6
    return True

# A list of the first few prime numbers to check against
TARGET_PRIMES = [
    2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47,
    53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109,
    113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179,
    181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241,
    251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313,
    317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389,
    397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461,
    463, 467, 479, 487, 491, 499, 503, 509, 521, 523, 541, 547,
    557, 563, 569, 571, 577, 587, 593, 599, 601, 607, 613, 617,
    619, 631, 641, 643, 647, 653, 659, 661, 673, 677, 683, 691,
    701, 709, 719, 727, 733, 739, 743, 751, 757, 761, 769, 773,
    787, 797, 809, 811, 821, 823, 827, 829, 839, 853, 857, 859,
    863, 877, 881, 883, 887, 907, 911, 919, 929, 937, 941, 947,
    953, 967, 971, 977, 983, 991, 997]

def evaluate_primes(individual, n_range=len(TARGET_PRIMES)):
    """
    Evaluates an individual equation tree for predicting prime gaps.
    Starts from the first prime and uses the GP's predicted gap to reach the next.
    Returns (correct_sequence_length, tree_size).
    """


    func = gp.compile(expr=individual, pset=pset)
    ordered_prime_count = 1

    tree_size = len(individual)
    current_prime = TARGET_PRIMES[0]

    for target_prime in TARGET_PRIMES[1:n_range]:
        try:
            # Calculate the gap using the function
            gap = func(current_prime)
            
            if abs (gap - round(gap)) < 1e-6:
                gap = int(round(gap))
            else:
                break

            next_prime = current_prime + gap
            # Check if the next prime is indeed prime and matches the target
            if  next_prime == target_prime:
                ordered_prime_count += 1
                current_prime = next_prime
            else:
                break

        except (ValueError, OverflowError, ZeroDivisionError):
            break
    return ordered_prime_count, tree_size
            

<span style = "font-size:30px;color:Yellow"> Creating Deap toolbox

In [ ]:
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, 0.5))
creator.create("individual", gp.PrimitiveTree, fitness=creator.FitnessMulti)
toolbox = base.Toolbox()
# How to generate one expression tree
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=5)
# How to create a single individual
toolbox.register("individual", tools.initIterate, creator.individual, toolbox.expr)
# --- ADD THIS LINE ---
# How to create a population from the individual generator
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
# How to compile a tree into a callable function
toolbox.register("compile", gp.compile, pset=pset)
# How to evaluate an individual
toolbox.register("evaluate", evaluate_primes, n_range=len(TARGET_PRIMES))
# how to select individuals for the next generation
toolbox.register("select", tools.selNSGA2)
# how to apply crossover
toolbox.register("mate", gp.cxOnePoint)
# how to apply mutation
toolbox.register("expr_mut", gp.genFull, min_=0, max_=5)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)
# how to apply the mutation operator
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=17))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=17))

<span style = "font-size:30px;color:Yellow"> Evolution Runs

In [34]:
import dill as pickle

# PATCH: store best individual for later analysis
def save_best_individual(best_ind, filename):
    with open(filename, "wb") as f:
        pickle.dump(best_ind, f)

def save_best_readable(best_ind, filename):
    with open(filename, "w") as f:
        f.write(str(best_ind))
        f.write("\nFitness: " + str(best_ind.fitness.values))

# PATCH: parsimony pressure wrapper
def evaluate_with_size_penalty(individual, max_size=20):
    prime_count, tree_size = evaluate_primes(individual)
    penalty = max(0, tree_size - max_size)  # penalise if size too large
    return (prime_count - penalty, tree_size)

# PATCH: reduce initial and mutation depths to avoid nested-parentheses error
toolbox.unregister("expr")
toolbox.unregister("expr_mut")
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)  # smaller initial trees
toolbox.register("expr_mut", gp.genFull, min_=0, max_=3)  # smaller mutations

# PATCH: stronger static limit
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=10))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=10))

# PATCH: swap evaluate function to penalised version
toolbox.unregister("evaluate")
toolbox.register("evaluate", evaluate_with_size_penalty)

euler_individual = creator.individual(euler_formula_tree)

stats_fit = tools.Statistics(lambda ind: ind.fitness.values)
stats_size = tools.Statistics(len)

mstats = tools.MultiStatistics(fitness=stats_fit, size=stats_size)
mstats.register("avg", np.mean)
mstats.register("std", np.std)
mstats.register("min", np.min)
mstats.register("max", np.max)

# PATCH: fail fast if tree is ridiculous before compiling
def safe_evaluate(individual):
    if len(individual) > 200:
        return (0, len(individual))
    return evaluate_with_size_penalty(individual)
toolbox.unregister("evaluate")
toolbox.register("evaluate", safe_evaluate)

# PATCHED run_with_logging with saving top performer
def run_with_logging(pop_size, toolbox, cx_prob, mut_prob, n_gen, stats, run_id):
    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

    pop = toolbox.population(n=pop_size)

    invalid_ind = [ind for ind in pop if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    if stats:
        record = stats.compile(pop)
        logbook.record(gen=0, nevals=len(invalid_ind), **record)
        print(f"Gen 0 | Best fitness: {record['fitness']['max']:.4f} | Avg fitness: {record['fitness']['avg']:.4f}")

    for gen in range(1, n_gen + 1):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < cx_prob:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if np.random.random() < mut_prob:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        pop[:] = offspring

        if stats:
            record = stats.compile(pop)
            logbook.record(gen=gen, nevals=len(invalid_ind), **record)
            if gen % 10 == 0 or gen == n_gen:
                print(f"Gen {gen}/{n_gen} | Best fitness: {record['fitness']['max']:.4f} | Avg fitness: {record['fitness']['avg']:.4f}| min fitness: {record['fitness']['min']:.4f}| max fitness: {record['fitness']['max']:.4f} | Size: {record['size']['avg']:.2f}")

    # PATCH: Save best individual after run
    best_ind = tools.selBest(pop, 1)[0]
    save_best_individual(best_ind, f"best_run_{run_id}.pkl")
    save_best_readable(best_ind, f"best_run_{run_id}.txt")

    return pop, logbook

pop_sizes = [1000]
num_generations = [100]     # , 200, 500]
crossover_probs = [0.7]     # , 0.8, 0.9]
mutation_probs = [0.01]     # , 0.05, 0.1]

# MAIN LOOP
if __name__ == "__main__":
    combos = list(itertools.product(pop_sizes, num_generations, crossover_probs, mutation_probs))
    total_runs = len(combos)

    for i, (pop_size, n_gen, cx_prob, mut_prob) in enumerate(combos, start=1):
        print(f"\n=== Run {i}/{total_runs} | pop_size={pop_size}, gens={n_gen}, cx={cx_prob}, mut={mut_prob} ===")
        pop, log = run_with_logging(pop_size, toolbox, cx_prob, mut_prob, n_gen, mstats, i)
        best_ind = tools.selBest(pop, 1)[0]
        print(f"Run {i} complete | Best fitness: {best_ind.fitness.values} | Individual: {best_ind}")


=== Run 1/1 | pop_size=1000, gens=100, cx=0.7, mut=0.01 ===
Gen 0 | Best fitness: 63.0000 | Avg fitness: 5.7975
Gen 10/100 | Best fitness: 157.0000 | Avg fitness: 5.5775| min fitness: -136.0000| max fitness: 157.0000 | Size: 18.28
Gen 20/100 | Best fitness: 247.0000 | Avg fitness: 5.4720| min fitness: -162.0000| max fitness: 247.0000 | Size: 19.99
Gen 30/100 | Best fitness: 253.0000 | Avg fitness: 5.5025| min fitness: -176.0000| max fitness: 253.0000 | Size: 21.83
Gen 40/100 | Best fitness: 309.0000 | Avg fitness: 6.6625| min fitness: -176.0000| max fitness: 309.0000 | Size: 23.76
Gen 50/100 | Best fitness: 651.0000 | Avg fitness: 7.2070| min fitness: -164.0000| max fitness: 651.0000 | Size: 25.44
Gen 60/100 | Best fitness: 641.0000 | Avg fitness: 7.6125| min fitness: -178.0000| max fitness: 641.0000 | Size: 26.79
Gen 70/100 | Best fitness: 687.0000 | Avg fitness: 8.7705| min fitness: -178.0000| max fitness: 687.0000 | Size: 27.89
Gen 80/100 | Best fitness: 1243.0000 | Avg fitness: 9.